# Data-centre waste heat recovery: results notebook

I use this notebook to examine the **outputs of the hourly model**, not to generate new assumptions after seeing the answer. All reference inputs are in `config.json`, and `run.py` produces the CSV results. Figures in the technical report are generated by the same code.

**Data status:** simulated IT load, demand, temperatures and prices, not recorded Copenhagen observations.

In [ ]:
from pathlib import Path
import sys, json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

ROOT = Path.cwd()
if not (ROOT / 'config.json').exists():
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT / 'src'))
summary = json.loads((ROOT / 'results' / 'summary.json').read_text())['metrics']
df = pd.read_csv(ROOT / 'results' / 'hourly_results.csv', parse_dates=['time_utc'])
cases = pd.read_csv(ROOT / 'results' / 'scenario_comparison.csv')
print('Hours:', len(df), 'Useful heat [MWh]:', round(summary['heat_delivered_mwh']))

## 1. Check the energy balance

The heat pump *upgrades* heat. Its condenser output is not simply the heat taken from IT cooling: compressor energy contributes as well. The useful network heat is smaller than condenser output because of delivery losses.

The balance residual below should be close to floating-point zero for every hour.

In [ ]:
thermal_residual = (df['source_heat_used_mw'] + df['compressor_electricity_mw']
                    - df['condenser_heat_mw'])
network_residual = (df['condenser_heat_mw'] - df['network_loss_mw']
                    - df['district_heat_delivered_mw'])
print('Max heat-pump residual [MW]:', abs(thermal_residual).max())
print('Max network residual [MW]:', abs(network_residual).max())
assert abs(thermal_residual).max() < 1e-9
assert abs(network_residual).max() < 1e-9

## 2. Seasonal demand matching

The data centre can produce heat all year, but only a fraction of that heat can be useful if the district-heating demand is lower in summer. This is why an annual energy-only estimate is inadequate.

In [ ]:
monthly = (df.set_index('time_utc')
           [['district_heat_demand_mwh', 'heat_delivery_mwh']]
           .resample('MS').sum())
ax = monthly.plot(kind='bar', figsize=(10,4))
ax.set_ylabel('MWh per month')
ax.set_xticklabels([x.strftime('%b') for x in monthly.index], rotation=0)
ax.set_title('Demand and recovered heat: synthetic reference year')
plt.tight_layout()

## 3. Temperature lift and the COP trade-off

A lower district-heating supply temperature improves COP. In a source-limited regime, it may nevertheless reduce total condenser heat because less compressor electricity is added to each MW of source heat. Heat quantity and efficiency should therefore be inspected together.

In [ ]:
selected = cases.set_index('case')[['heat_delivered_mwh','mean_running_cop','npv_eur']]
display(selected.round(2))
plt.figure(figsize=(7,4))
plt.scatter(df['district_supply_c']-df['source_supply_c'], df['cop'], s=2, alpha=.2)
plt.xlabel('External temperature lift [K]')
plt.ylabel('Model COP [-]')
plt.title('COP versus network temperature lift')
plt.tight_layout()

## 4. Economic assumptions matter

The positive NPV of the reference case is conditional on the assumed displaced boiler, prices, CAPEX and real discount rate. The 280-point sweep is a *stress test*, not a statistical uncertainty model. No frequency from that sweep should be described as the actual probability of success.

In [ ]:
sweep = pd.read_csv(ROOT / 'results' / 'assumption_sweep.csv')
print('Illustrative NPV range [EUR]:',
      int(sweep['npv_eur'].min()), 'to', int(sweep['npv_eur'].max()))
plt.figure(figsize=(7.5,4))
plt.scatter(sweep['heat_delivered_mwh']/1000, sweep['npv_eur']/1e6, s=12, alpha=.5)
plt.axhline(0,color='black',linestyle='--',linewidth=1)
plt.xlabel('Delivered heat [GWh/year]')
plt.ylabel('NPV [million EUR]')
plt.title('Parameter-space exploration (not a probabilistic forecast)')
plt.tight_layout()

## 5. What I would measure next

A technically serious next step would need three real input series: hourly source-cooling temperature and flow, a nearby district network's hourly heat demand and temperatures, and a manufacturer performance map for the chosen heat pump. Then I could test model predictions on a separate observation period rather than relying only on internal equation checks.

The full model and equations are described in `report/technical_report.pdf`, with independent hand checks in `docs/VALIDATION.md`.